# 🐣 Document Intelligence with Docling: Unlocking Complex Academic Content

This notebook demonstrates **Document Intelligence** - the advanced capability to understand and process complex documents like research papers, academic materials, and structured content that traditional RAG systems struggle with.

**The Challenge:**
Imagine trying to build an educational AI assistant using only basic text extraction from research papers. You'd lose:
- **📊 Table data** with crucial research findings
- **🧮 Mathematical formulas** and scientific notation  
- **📈 Charts and figures** that provide key insights
- **🏛️ Document structure** like sections, references, and metadata
- **📝 Multi-column layouts** common in academic papers

**The Solution: Docling**
Docling is an advanced document processing toolkit that can turn these advanced content structures into normal plaintext, extracting text out from images, PDFs, tables, etc.

**What You'll Build:**
- **🔬 Intelligent Document Processor**: Extract rich content from complex PDFs
- **📚 Enhanced RAG System**: Query tables, formulas, and structured content  
- **🎯 Academic AI Assistant**: Answer questions using complete document understanding
- **⚡ Production Pipeline**: Handle real-world educational materials at scale

#### Let's build document intelligence that truly understands academic content! 🚀

## 📚 Import Libraries for Document Intelligence

Import the essential libraries for building our document intelligence RAG system with Docling processing capabilities.

In [ ]:
# Core libraries
import requests
import json
import os
import sys
sys.path.append('..')

# OpenAI-compatible client for direct LLM access
from openai import OpenAI

# Vector database and embeddings
from pymilvus import connections, utility, Collection, CollectionSchema, FieldSchema, DataType
from sentence_transformers import SentenceTransformer

# Display utilities
from termcolor import cprint

## 🔗 Connect to Llama 3.2 and Milvus

‼️⚠️ IMPORTANT ⚠️‼️

Add your **username** and **cluster domain** below to connect to your Milvus instance.

In [ ]:
# IMPORTANT! Add your username and cluster domain here
username = "<USER_NAME>"
cluster_domain = "<CLUSTER_DOMAIN>"

# --- LLM configuration ---
LLM_ENDPOINT = "http://llama-32-predictor.ai501.svc.cluster.local:8080"
MODEL_NAME = "llama32"

client = OpenAI(
    base_url=LLM_ENDPOINT + "/v1",
    api_key="no-key-required",
)

temperature = 0.0
max_tokens = 512

print(f"✅ LLM endpoint: {LLM_ENDPOINT}")
print(f"   Model: {MODEL_NAME}")

# --- Milvus configuration ---
collection_name = "docling_rag_collection"
embedding_model_name = "all-MiniLM-L6-v2"
embedding_dim = 384

connections.connect(
    uri=f"http://milvus.{username}-canopy.svc.cluster.local:19530",
    alias="default"
)

print(f"✅ Connected to Milvus at milvus.{username}-canopy.svc.cluster.local:19530")

## 🔬 Docling Processing Function Implementation

This function connects to the Docling service and processes documents through the intelligent pipeline we just described.  
Let's test Docling's document intelligence on a complex academic paper. We'll use a real research paper that contains tables, mathematical formulas, and figures 📈

> Note: Because Docling is using more advanced document extraction it usually takes a little bit to extract the information.  
This particular pdf we use should take 🕒 3-4 minutes depending on the service capacity. Grab some ☕ coffee!

In [ ]:
def docling_processing(url):

    # Connect to the deployed Docling service in the cluster
    api_address = "http://docling-v0-7-0-predictor.ai501.svc.cluster.local:5001"
    headers = {"Content-Type": "application/json"}

    print(f"🔗 Connecting to Docling service at {api_address}")
    
    # Docling settings for processing
    payload = {
        "http_sources": [{"url": url}],
        "options": {
            "to_formats": ["md"],
            "image_export_mode": "placeholder"
        },
    }
    
    try:
        # Send document to Docling for intelligent analysis
        response = requests.post(
            f"{api_address}/v1alpha/convert/source",
            json=payload,
            headers=headers,
            timeout=300
        )
        
        response.raise_for_status()
        
        result_data = response.json()
        md_content = result_data["document"]["md_content"]
        
        return md_content
        
    except requests.exceptions.Timeout:
        print(f"⏰ Processing timeout - complex documents may need more time")
        raise
    except requests.exceptions.RequestException as e:
        print(f"❌ Docling processing failed: {e}")
        raise
    except KeyError as e:
        print(f"❌ Unexpected response format: {e}")
        raise

In [ ]:
# We choose a research paper called "First Mapping the Canopy Height of Primeval Forests in the Tallest Tree Area of Asia"
url = "https://arxiv.org/pdf/2404.14661"

md_content = docling_processing(url)

print(f"\n🎉 Document intelligence processing complete!")
print(f"📊 Content preview (first 500 characters):")
print(f"{'='*60}")
print(md_content[:500] + "..." if len(md_content) > 500 else md_content)
print(f"{'='*60}")
print(f"📈 Total processed content: {len(md_content)} characters")
print(f"📝 Docling has extracted and structured the complete document content!")

## 📃 Using the processed document for RAG

We now chunk the Docling-processed markdown, embed each chunk with `all-MiniLM-L6-v2`, and store everything in our Milvus vector database — the same database you saw in notebook 2.

In [ ]:
# Helper to split the markdown into overlapping word-level chunks
def chunk_text(text, chunk_size=512, overlap=50):
    words = text.split()
    chunks = []
    step = chunk_size - overlap
    for i in range(0, len(words), step):
        chunk = " ".join(words[i : i + chunk_size])
        if chunk:
            chunks.append(chunk)
    return chunks

chunks = chunk_text(md_content)
print(f"📄 Split document into {len(chunks)} chunks")

In [ ]:
# Set up Milvus collection for the docling RAG chunks
if utility.has_collection(collection_name):
    utility.drop_collection(collection_name)
    print(f"🧹 Dropped existing collection '{collection_name}'")

id_field = FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=False)
embedding_field = FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=embedding_dim)

schema = CollectionSchema(
    fields=[id_field, embedding_field],
    description="Docling-processed document chunks",
    enable_dynamic_field=False
)

collection = Collection(
    name=collection_name,
    schema=schema,
    using="default",
    shards_num=2,
    consistency_level="Strong"
)

print(f"✅ Created Milvus collection: {collection_name}")

In [ ]:
# Embed all chunks and insert them into Milvus
embed_model = SentenceTransformer(embedding_model_name)

print(f"🔄 Embedding {len(chunks)} chunks with '{embedding_model_name}'...")
embeddings = embed_model.encode(chunks, show_progress_bar=True)

# Keep a local id→text map for retrieval later
id_to_chunk = {i: chunk for i, chunk in enumerate(chunks)}

data = [{"id": i, "embedding": vec.tolist()} for i, vec in enumerate(embeddings)]
collection.insert(data=data)

collection.create_index(
    field_name="embedding",
    index_params={
        "metric_type": "COSINE",
        "index_type": "IVF_FLAT",
        "params": {"nlist": 128}
    },
    index_name="idx"
)

collection.flush()
collection.load()

print(f"\n✅ Document ingestion complete!")
print(f"🎯 {len(chunks)} Docling-processed chunks are now searchable via semantic similarity!")

Now that we have ingested the document (added it into our Vector Database) we can query for it just like we did before!  
Feel free to play around with different queries to see what it answers.

In [ ]:
# Test queries for the processed document
queries = [
    "What is the PRFXception?",
    "The accuracy values of overall model prediction and residual cross-validation for five regions in southeast Tibet and four regions in northwest Yunnan"
]

for prompt in queries:
    cprint(f"\nUser> {prompt}", "blue")

    # Embed the query and search Milvus for the most relevant chunks
    query_embedding = embed_model.encode([prompt])

    results = collection.search(
        data=query_embedding,
        anns_field="embedding",
        param={"metric_type": "COSINE"},
        limit=5,
        output_fields=[]
    )

    cprint(f"\n--- RAG Metadata ---", "yellow")
    retrieved_chunks = []
    for match in results[0]:
        cprint(f"  chunk_id={match.id}  score={match.score:.4f}", "cyan")
        retrieved_chunks.append(id_to_chunk[match.id])

    # Build the prompt with retrieved context
    messages = [{"role": "system", "content": "You are a helpful assistant."}]
    prompt_context = "\n\n".join(retrieved_chunks)
    extended_prompt = (
        f"Please answer the given query using the context below.\n\n"
        f"CONTEXT:\n{prompt_context}\n\nQUERY:\n{prompt}"
    )
    messages.append({"role": "user", "content": extended_prompt})

    # Call Llama 3.2 directly via the OpenAI-compatible endpoint
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        max_tokens=max_tokens,
        temperature=temperature,
        stream=True,
    )

    cprint("inference> ", color="magenta", end="")
    for chunk in response:
        if chunk.choices and chunk.choices[0].delta.content:
            cprint(chunk.choices[0].delta.content, color="magenta", end="")
    print()  # newline after streamed output

In [ ]:
# Clean up - release collection from memory
collection.release()
utility.drop_collection(collection_name)  # delete the collection

## 🎉 You have used Docling to enhance your document processing!

Your document intelligence system can now understand and query the most complex academic content - transforming how educational institutions handle knowledge discovery and research! 🚀  
Go back to the instructions to see how we can automate our document ingestion.